# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [55]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

tablas = pd.read_sql(
    "SELECT name FROM SQLITE_MASTER WHERE type='table'",
    conexion
)
print (tablas)

pacientes = pd.read_sql(

"PRAGMA table_info(pacientes)",

conexion
)
medicos = pd.read_sql(
"PRAGMA table_info(medicos)",
conexion
)
turnos = pd.read_sql(
"PRAGMA table_info(turnos)",

conexion
)
df_pacientes = pd.read_sql("SELECT * FROM pacientes",conexion)
df_medicos = pd.read_sql("SELECT * FROM medicos",conexion)
df_turnos = pd.read_sql("SELECT * FROM turnos",conexion)

        name
0  pacientes
1    medicos
2     turnos


In [47]:
df_pacientes = pd.read_sql("SELECT * FROM pacientes",conexion)
df_pacientes.shape

(66, 6)

In [48]:
df_medicos = pd.read_sql("SELECT * FROM medicos",conexion)
df_medicos.shape

(20, 5)

In [49]:
df_turnos = pd.read_sql("SELECT * FROM turnos",conexion)
df_turnos.shape

(210, 8)

**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

TABLAS: PACIENTES MEDICOS TURNOS
Dentro de la tABLA pacientes encontramos las siguientes Columnas: id(enter), nombre(string), dni(string), fecha_nacimiento(string), ciudad, obra_social

En la tabla medicos encontrarmos las siguientes columnas: id, nombre(string), especialidad(string), matricula(string) y activo(string)

En la tabla turnos encontramos las siguientes columnas: id(entero), paciente_id(entero), medico_id(entero), fecha(string), hora(string), motivo(string), costo(string), estado(string)

La tabla pacientes tiene 66 filas/registros
La tabla medicos tiene 20 filas/registros
La tabla turnos tiene 210 filas/registros


### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [3]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")
# Leer toda la tabla
df = pd.read_sql("SELECT * FROM pacientes", conexion)
nulos_columna = df.isnull().sum()
print(nulos_columna)
porcentaje_nulos = (df.isna().sum() / len(df)) * 100
print(porcentaje_nulos)
filas_duplicadas = df[df.duplicated(subset=df.columns.drop("id"))]
print(filas_duplicadas)
nulos_dni = df[df["dni"].isnull()]
print(nulos_dni)


id                   0
nombre               0
dni                  7
fecha_nacimiento    11
ciudad               8
obra_social         12
dtype: int64
id                   0.000000
nombre               0.000000
dni                 10.606061
fecha_nacimiento    16.666667
ciudad              12.121212
obra_social         18.181818
dtype: float64
    id          nombre         dni fecha_nacimiento        ciudad  \
3    4  Martina García  28.341.992       1985-04-12  Buenos Aires   
10  11    Roberto Díaz  31.882.441       1990-11-30       Córdoba   

      obra_social  
3            OSDE  
10  Swiss Medical  
    id          nombre  dni fecha_nacimiento         ciudad    obra_social
7    8      Ana Romero  NaN       1988-06-14        Córdoba  Swiss Medical
27  28     Matías Vera  NaN       1967-03-27        Tucumán         Medife
33  34    Belén Suárez  NaN       1969-11-19            NaN            NaN
35  36   Franco Medina  NaN              NaN  Mar del Plata           PAMI
56  57   Fr

In [6]:
df = pd.read_sql("SELECT * FROM medicos", conexion)
nulos_columna = df.isnull().sum()
print(nulos_columna)
porcentaje_nulos = (df.isna().sum() / len(df)) * 100
print(porcentaje_nulos)
filas_duplicadas = df[df.duplicated(subset=df.columns.drop("id"))]
print(filas_duplicadas)
nulos_matricula = df[df["matricula"].isnull()]
print(nulos_matricula)
nulos_activo = df[df["activo"].isnull()]
print(nulos_activo)


id              0
nombre          0
especialidad    0
matricula       2
activo          4
dtype: int64
id               0.0
nombre           0.0
especialidad     0.0
matricula       10.0
activo          20.0
dtype: float64
   id             nombre    especialidad matricula activo
4   5  Dra. Beatriz Sosa  Clínica Médica  MP-12345      1
    id               nombre especialidad matricula activo
2    3  Dra. Carmen Aguirre  Cardiología       NaN      1
17  18   Dra. Gabriela Luna  Psiquiatría       NaN      1
    id              nombre    especialidad matricula activo
6    7  Dra. Laura Giménez       Pediatría  MP-67890    NaN
8    9    Dr. Nicolás Vera  Endocrinología  MP-46953    NaN
11  12  Dr. Sebastián Ríos    Dermatología  MP-53113    NaN
19  20   Dra. Silvina Vega     Psiquiatría  MP-30349    NaN


In [7]:
df = pd.read_sql("SELECT * FROM turnos", conexion)
nulos_columna = df.isnull().sum()
print(nulos_columna)
porcentaje_nulos = (df.isna().sum() / len(df)) * 100
print(porcentaje_nulos)
filas_duplicadas = df[df.duplicated(subset=df.columns.drop("id"))]
print(filas_duplicadas)
nulos_motivo = df[df["motivo"].isnull()]
print(nulos_motivo)


id              0
paciente_id     0
medico_id       0
fecha           0
hora           16
motivo         16
costo           9
estado          0
dtype: int64
id             0.000000
paciente_id    0.000000
medico_id      0.000000
fecha          0.000000
hora           7.619048
motivo         7.619048
costo          4.285714
estado         0.000000
dtype: float64
    id  paciente_id  medico_id       fecha   hora         motivo costo  \
10  11            1          1  2024-03-01  09:00  Control anual  3500   

       estado  
10  realizado  
      id  paciente_id  medico_id       fecha   hora motivo costo       estado
14    15           14          1  2024-03-07  09:00    NaN  3500    realizado
23    24            1         15  2024-03-09  16:00    NaN   N/A    realizado
74    75           48         17  2024-03-22  12:15    NaN  3500      AUSENTE
81    82           66          2  2024-03-06  09:30    NaN   N/A    realizado
105  106           33         10  2024-03-07  13:30    NaN  5500 

Tabla pacientes 
La columna dni tiene 7 valores nulos, la columna fecha_nacimiento tiene 11 valores nulos, la columna ciudad tiene 8 valores nulos y la columna obra_social tiene 12 valores nulos, representando el 10,60% en dni, 16,66% en fecha de nacimiento, 12,12% en ciudad y 18,18 en obra social. Tiene 2 filas duplicadas 
3    4  Martina García  28.341.992       1985-04-12  Buenos Aires   
10  11    Roberto Díaz  31.882.441       1990-11-30       Córdoba   
No se encontraron datos erroneos que nos impidan poder trabajar con los registros de la tabla
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Tabla medicos
La columna matricula tiene 2 valores nulos, y la columnna activo 4 valores nulos, a su vez representan el 10% de la matricula y 20% de la columna activo. Se encontro una fila duplicada
4   5  Dra. Beatriz Sosa  Clínica Médica  MP-12345      1
No se encontraron datos erroneos que nos impidan poder trabajar con los registros de la tabla
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Tabla turnos
La columna hora tiene 16 valores nulos,la columna motivo 16 valores nulos, la columna costo tiene 9 valores nulos a su vez representan el 7.6% en hora 7.6% en motivo, y el 4.2% en costo. Se encontró una fila duplicada 
    id  paciente_id  medico_id       fecha   hora         motivo costo  \
10  11            1          1  2024-03-01  09:00  Control anual  3500   
No se encontraron datos erroneos que nos impidan poder trabajar con los registros de la tabla

Se encontraron datos faltantes en la columna costo de la tabla turnos pero no necesariamente erroneos

Listado de problemas encontrados en las tablas
1 Tabla pacientes Columna dni 7 valores nulos
2 Tabla pacientes Columna fecha_nacimiento 11 valores nulos
3 Tabla pacientes Columna ciudad 8 valores nulos
4 Tabla pacientes Columna obra_social 12 valores nulos
5 Tabla pacientes 2 filas duplicadas
6 Tabla medicos columna matricula 2 valores nulos
7 Tabla medicos columna activo 4 valores nulos
8 Tabla medicos 1 fila duplicada
9 Tabla turnos columna hora 16 valores nulos
10 Tabla turnos columna motivo 16 valores nulos
11 Tabla turnos columna costo 9 valores nulos
12 Tabla turnos 1 fila duplicada



**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | ciudad | 8 nulos | relleno el campo (con no informada) | No es critico para poder tener un registro del paciente |
| pacientes | dni | 7 nulos | eliminar registros |El DNI identifica de manera única al paciente, se elimina ya que no es posible tener un registro unico del paciente |
| pacientes | fecha_nacimiento | 11 nulos | reemplazar por NULL | Son el 16% no los desctaria ya que son una proporcion alta |
| pacientes | obra_social | 12 nulos | reemplazar por NULL | Son el 18% no los desctaria ya que son una proporcion alta |
| pacientes | nombre dni fecha_nacimiento ciudad obra_social | 2 filas duplicadas | elimino el registro duplicado | guardo el registro original y verifico que no tenga una foreing key que impacte en otra tabla |
| medicos | matricula | 2 valores nulos | reemplazar por NULL | tratar de averiguar la matricula de los medicos pero no eliminar el registro, porque puede dificultar la trazabilidad y una auditoria |
| medicos | activo | 4 valores nulos | reemplazar por False o NULL |  si son médicos que ya no pertenecen a la empresa los actualizo a FALSE si no tengo info los trato como NULL |
| medicos | id nombre especialidad matricula activo | 1 fila duplicada | eliminar registro de la fila duplicada | guardo el registro original y verifico que no tenga una foreing key que impacte en otra tabla |
| turnos | hora | 16 valores nulos | le crearia un horario ficticio | le agrego un horario 00:00 para saber fueron turnos con valores nulos
| turnos | motivo | 16 valores nulos | le creo un motivo ficticio | le agrego un motivo que sea "sin motivo" para no tratarlo como nulo
| turnos | costo | 9 valores nulos | reemplazo por valores de esos estudios | buscaria un valor teorico para reemplazar el dato nulo pero no lo utilizaria para descuentos ni para facturación solo para limpieza |
| turnos | paciente_id  medico_id fecha hora motivo costo | 1 fila duplicada | elimino registro de la fila duplicada | guardo el registro original y verifico que no tenga una foreing key que impacte en otra tabla |

### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [ ]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()
# Tu código acá

# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()
# Tu código acá

# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
# Tu código acá

> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [ ]:
# tu codigo aquí

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [ ]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

# tu reporte acá

→ [Ir a la solución](./solucion.md)